# Lezione 13 — Deep learning nel browser con TensorFlow.js

**Obiettivi di oggi**
- Addestrare un modello semplice direttamente nel browser.
- Spiegare come convertire un modello addestrato in Python per usarlo in JavaScript.
- Relazionare entrambi i passaggi con il caso di un'IA che gioca a Pong.

Consulta il [capitolo 10 del manuale](../manual/10_tensorflowjs.md) per la spiegazione completa.

## 1. Addestra un modello direttamente nel browser

La siguiente celda renderiza HTML/JavaScript real: se ejecuta en tu navegador (dentro de esta misma página de Colab/Jupyter), entrena una regresión lineal en TensorFlow.js, y muestra el resultado. Abre la consola de desarrollador (F12) para ver más detalle si algo no se muestra.

In [ ]:
%%html
<script src="https://cdn.jsdelivr.net/npm/@tensorflow/tfjs"></script>
<div id="resultado">Entrenando en el navegador...</div>
<script>
  const modelo = tf.sequential();
  modelo.add(tf.layers.dense({units: 1, inputShape: [1]}));
  modelo.compile({optimizer: "sgd", loss: "meanSquaredError"});

  const xs = tf.tensor2d([1, 2, 3, 4], [4, 1]);
  const ys = tf.tensor2d([2, 4, 6, 8], [4, 1]);

  modelo.fit(xs, ys, {epochs: 200}).then(() => {
    const prediccion = modelo.predict(tf.tensor2d([5], [1, 1]));
    prediccion.data().then(valores => {
      document.getElementById("resultado").innerText =
        "Predicción para 5 (se espera cerca de 10): " + valores[0].toFixed(2);
    });
  });
</script>

## 2. Addestra in Python, converti per il browser

Addestriamo un modello di Keras noto (come quello del capitolo 1) e lo convertiamo nel formato che TensorFlow.js può caricare.

In [ ]:
import tensorflow as tf

(x_entreno, y_entreno), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_entreno, x_test = x_entreno / 255.0, x_test / 255.0

modelo = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax"),
])
modelo.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
modelo.fit(x_entreno, y_entreno, epochs=2)
modelo.save("modelo_mnist.h5")

In [ ]:
!pip install -q tensorflowjs
!tensorflowjs_converter --input_format=keras modelo_mnist.h5 modelo_web/
!ls modelo_web/

La cartella `modelo_web/` contiene ora un `model.json` e i pesi in formato binario. Questi file sono quelli che caricheresti nella tua pagina con `tf.loadLayersModel("modelo_web/model.json")`, come mostra il capitolo 10 del manuale.


## 3. Il caso di un'IA che gioca a Pong

Il progetto originale di questa lezione addestra una rete che riceve come input la posizione della palla e della racchetta, e decide di muovere la racchetta su, giù o lasciarla ferma — lo stesso approccio di apprendimento per rinforzo del capitolo 6, qui distribuito nel browser invece che in Python. Il pattern è identico a quello che hai appena praticato: addestrare in Python, convertire il modello, e caricarlo con `tf.loadLayersModel` in una pagina HTML dove l'utente può giocare contro l'IA.


## Riepilogo

- Entrenaste un modelo directamente en el navegador con TensorFlow.js.
- Convertiste un modelo de Keras a un formato que el navegador puede cargar con `tensorflowjs_converter`.
- Un juego con IA, como el Pong, sigue exactamente este mismo patrón de entrenar en Python y desplegar en el navegador.

Continua con il [notebook de práctica di questa lezione](../notebooks_practica/clase13_practica_tensorflowjs.ipynb).